In [24]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [25]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [26]:
import shutil
from pathlib import Path

DRIVE_DATA = Path("/content/drive/MyDrive/bone_tumor_data/final")
LOCAL_DATA = Path("/content/bone_tumor_data/final")

if LOCAL_DATA.exists():
    shutil.rmtree(LOCAL_DATA)

shutil.copytree(DRIVE_DATA, LOCAL_DATA)

print("Dataset copied successfully.")
print("Dataset location:", LOCAL_DATA)

Dataset copied successfully.
Dataset location: /content/bone_tumor_data/final


In [27]:
from pathlib import Path

DATA_ROOT = Path("/content/bone_tumor_data/final")

print("Dataset path:", DATA_ROOT)
print("Train exists:", (DATA_ROOT / "train").exists())
print("Valid exists:", (DATA_ROOT / "valid").exists())
print("Test exists:", (DATA_ROOT / "test").exists())

Dataset path: /content/bone_tumor_data/final
Train exists: True
Valid exists: True
Test exists: True


In [28]:
import pandas as pd

for split in ["train", "valid", "test"]:
    image_dir = DATA_ROOT / split / "images"
    metadata_file = DATA_ROOT / split / "metadata.csv"

    image_count = len(list(image_dir.glob("*.png")))
    metadata_count = len(pd.read_csv(metadata_file))

    print(
        f"{split}: "
        f"{image_count} images, "
        f"{metadata_count} metadata rows"
    )

train: 10052 images, 10052 metadata rows
valid: 1084 images, 1084 metadata rows
test: 1067 images, 1067 metadata rows


Data Augmentation

To evaluate the effect of data augmentation, ColorJitter and Gaussian noise were applied only to the training images. Validation and test images were kept unchanged. ImageNet normalization was applied consistently to all splits because the model uses ImageNet pretrained weights


In [35]:
training_transform = transforms.Compose([
    transforms.ColorJitter(
        brightness=0.10,
        contrast=0.10
    ),
    transforms.ToTensor(),
    AddGaussianNoise(
        standard_deviation=0.01
    ),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

valid_test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

print("Corrected augmentation transforms created.")

Corrected augmentation transforms created.


In [36]:
train_dataset = BoneTumorDataset(
    split="train",
    transform=training_transform
)

valid_dataset = BoneTumorDataset(
    split="valid",
    transform=valid_test_transform
)

test_dataset = BoneTumorDataset(
    split="test",
    transform=valid_test_transform
)

print("Train:", len(train_dataset))
print("Valid:", len(valid_dataset))
print("Test:", len(test_dataset))

Train: 10052
Valid: 1084
Test: 1067


In [37]:
BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

print("DataLoaders recreated successfully.")

DataLoaders recreated successfully.


In [38]:
model = models.resnet50(
    weights=models.ResNet50_Weights.DEFAULT
)

model.fc = nn.Linear(
    in_features=2048,
    out_features=2
)

for param in model.parameters():
    param.requires_grad = False

for param in model.fc.parameters():
    param.requires_grad = True

model = model.to(device)

print("Fresh ResNet50 model created.")
print("Trainable parameters:",
      sum(p.numel() for p in model.parameters() if p.requires_grad))

Fresh ResNet50 model created.
Trainable parameters: 4098


In [39]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.fc.parameters(),
    lr=0.001
)

print("Fresh optimizer created.")

Fresh optimizer created.


In [40]:
NUM_EPOCHS = 3

best_val_accuracy = 0.0

best_model_path = (
    "/content/drive/MyDrive/"
    "resnet50_imagenet_augmentation_corrected_best.pth"
)

train_losses = []
train_accuracies = []
valid_losses = []
valid_accuracies = []

for epoch in range(NUM_EPOCHS):

    train_loss, train_accuracy = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        device
    )

    valid_loss, valid_accuracy = validate_one_epoch(
        model,
        valid_loader,
        criterion,
        device
    )

    train_losses.append(train_loss)
    train_accuracies.append(train_accuracy)

    valid_losses.append(valid_loss)
    valid_accuracies.append(valid_accuracy)

    print(
        f"Epoch {epoch + 1}/{NUM_EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_accuracy:.4f} | "
        f"Valid Loss: {valid_loss:.4f} | "
        f"Valid Acc: {valid_accuracy:.4f}"
    )

    if valid_accuracy > best_val_accuracy:
        best_val_accuracy = valid_accuracy

        torch.save(
            model.state_dict(),
            best_model_path
        )

        print("  → Best model saved.")

print("\nTraining complete.")
print(f"Best validation accuracy: {best_val_accuracy:.4f}")
print(f"Best model saved at: {best_model_path}")

Epoch 1/3 | Train Loss: 0.3403 | Train Acc: 0.8641 | Valid Loss: 0.5506 | Valid Acc: 0.7205
  → Best model saved.
Epoch 2/3 | Train Loss: 0.2711 | Train Acc: 0.8892 | Valid Loss: 0.5313 | Valid Acc: 0.7445
  → Best model saved.
Epoch 3/3 | Train Loss: 0.2561 | Train Acc: 0.8975 | Valid Loss: 0.5271 | Valid Acc: 0.7463
  → Best model saved.

Training complete.
Best validation accuracy: 0.7463
Best model saved at: /content/drive/MyDrive/resnet50_imagenet_augmentation_corrected_best.pth


In [41]:
# Load the best augmentation model
model.load_state_dict(
    torch.load(
        best_model_path,
        map_location=device
    )
)

model = model.to(device)
model.eval()

print("Best augmentation model loaded.")
print(f"Best validation accuracy: {best_val_accuracy:.4f}")

Best augmentation model loaded.
Best validation accuracy: 0.7463


In [42]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

all_labels = []
all_predictions = []
all_probabilities = []

model.eval()

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)

        outputs = model(images)

        probabilities = torch.softmax(outputs, dim=1)
        predictions = outputs.argmax(dim=1)

        all_labels.extend(labels.numpy())
        all_predictions.extend(predictions.cpu().numpy())
        all_probabilities.extend(
            probabilities[:, 1].cpu().numpy()
        )

accuracy = accuracy_score(
    all_labels,
    all_predictions
)

precision = precision_score(
    all_labels,
    all_predictions
)

recall = recall_score(
    all_labels,
    all_predictions
)

f1 = f1_score(
    all_labels,
    all_predictions
)

auc = roc_auc_score(
    all_labels,
    all_probabilities
)

cm = confusion_matrix(
    all_labels,
    all_predictions
)

tn, fp, fn, tp = cm.ravel()

specificity = tn / (tn + fp)

print("Final Test Results — ResNet50 + ImageNet + Augmentation")
print("-" * 60)
print(f"Accuracy:    {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"Precision:   {precision:.4f} ({precision*100:.2f}%)")
print(f"Recall:      {recall:.4f} ({recall*100:.2f}%)")
print(f"Specificity: {specificity:.4f} ({specificity*100:.2f}%)")
print(f"F1 Score:    {f1:.4f} ({f1*100:.2f}%)")
print(f"ROC-AUC:     {auc:.4f} ({auc*100:.2f}%)")

print("\nConfusion Matrix:")
print(cm)

print("\nTN:", tn)
print("FP:", fp)
print("FN:", fn)
print("TP:", tp)

Final Test Results — ResNet50 + ImageNet + Augmentation
------------------------------------------------------------
Accuracy:    0.7451 (74.51%)
Precision:   0.6600 (66.00%)
Recall:      0.7170 (71.70%)
Specificity: 0.7631 (76.31%)
F1 Score:    0.6874 (68.74%)
ROC-AUC:     0.8184 (81.84%)

Confusion Matrix:
[[496 154]
 [118 299]]

TN: 496
FP: 154
FN: 118
TP: 299
